In [ ]:
# ========================================================================
# MODEL COMPARISON: XGBoost vs Random Forest vs LightGBM
# ========================================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 70)
print("MODEL COMPARISON ANALYSIS")
print("=" * 70)

# Load all model summaries
try:
    xgb_summary = pd.read_csv('model_performance_summary_tuned.csv')
    rf_summary = pd.read_csv('rf_model_summary.csv')
    lgb_summary = pd.read_csv('lgb_model_summary.csv')

    print("\n✓ All model summaries loaded")
except FileNotFoundError as e:
    print(f"\n❌ Error: {e}")
    print("Please run all three model training scripts first!")
    exit()

# Create comparison table
comparison = pd.DataFrame({
    'Model': ['XGBoost', 'Random Forest', 'LightGBM'],
    'Test_R2': [
        float(xgb_summary[xgb_summary['Metric'] == 'XGBoost Tuned R²']['Value'].values[0]),
        float(rf_summary[rf_summary['Metric'] == 'Test R²']['Value'].values[0]),
        float(lgb_summary[lgb_summary['Metric'] == 'Test R²']['Value'].values[0])
    ],
    'Test_RMSE_M': [
        float(xgb_summary[xgb_summary['Metric'] == 'Test RMSE (₹M)']['Value'].values[0]),
        float(rf_summary[rf_summary['Metric'] == 'Test RMSE (₹M)']['Value'].values[0]),
        float(lgb_summary[lgb_summary['Metric'] == 'Test RMSE (₹M)']['Value'].values[0])
    ],
    'Train_Test_Gap': [
        float(xgb_summary[xgb_summary['Metric'] == 'Train-Test Gap']['Value'].values[0]),
        float(rf_summary[rf_summary['Metric'] == 'Train-Test Gap']['Value'].values[0]),
        float(lgb_summary[lgb_summary['Metric'] == 'Train-Test Gap']['Value'].values[0])
    ]
})

print("\n" + "=" * 70)
print("PERFORMANCE COMPARISON")
print("=" * 70)
print("\n")
print(comparison.to_string(index=False))

# Rank models
comparison['R2_Rank'] = comparison['Test_R2'].rank(ascending=False).astype(int)
comparison['RMSE_Rank'] = comparison['Test_RMSE_M'].rank(ascending=True).astype(int)
comparison['Gap_Rank'] = comparison['Train_Test_Gap'].rank(ascending=True).astype(int)
comparison['Overall_Rank'] = (comparison['R2_Rank'] + comparison['RMSE_Rank'] + comparison['Gap_Rank']) / 3

print("\n" + "=" * 70)
print("MODEL RANKINGS (1 = Best)")
print("=" * 70)
print("\n")
print(comparison[['Model', 'R2_Rank', 'RMSE_Rank', 'Gap_Rank', 'Overall_Rank']].to_string(index=False))

# Determine winner
best_model = comparison.loc[comparison['Overall_Rank'].idxmin(), 'Model']

print("\n" + "=" * 70)
print("WINNER")
print("=" * 70)
print(f"\n🏆 Best Model: {best_model}")

# Visualization: Comparison Bar Chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# R² Score
axes[0].bar(comparison['Model'], comparison['Test_R2'], color=['steelblue', 'forestgreen', 'purple'])
axes[0].set_title('Test R² Score (Higher = Better)', fontweight='bold')
axes[0].set_ylabel('R² Score')
axes[0].set_ylim([comparison['Test_R2'].min() - 0.05, comparison['Test_R2'].max() + 0.05])

# RMSE
axes[1].bar(comparison['Model'], comparison['Test_RMSE_M'], color=['steelblue', 'forestgreen', 'purple'])
axes[1].set_title('Test RMSE (Lower = Better)', fontweight='bold')
axes[1].set_ylabel('RMSE (₹ Millions)')

# Train-Test Gap
axes[2].bar(comparison['Model'], comparison['Train_Test_Gap'], color=['steelblue', 'forestgreen', 'purple'])
axes[2].set_title('Train-Test Gap (Lower = Better)', fontweight='bold')
axes[2].set_ylabel('Gap')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: model_comparison.png")
plt.close()

# Save comparison
comparison.to_csv('model_comparison_summary.csv', index=False)
print("✓ Saved: model_comparison_summary.csv")

print("\n" + "=" * 70)
print("✓ COMPARISON COMPLETE!")
print("=" * 70)

MODEL COMPARISON ANALYSIS

✓ All model summaries loaded

PERFORMANCE COMPARISON


        Model  Test_R2  Test_RMSE_M  Train_Test_Gap
      XGBoost   0.3135      1001.89          0.0480
Random Forest   0.3166      1005.60          0.1362
     LightGBM   0.3027      1005.31          0.0104

MODEL RANKINGS (1 = Best)


        Model  R2_Rank  RMSE_Rank  Gap_Rank  Overall_Rank
      XGBoost        2          1         2      1.666667
Random Forest        1          3         3      2.333333
     LightGBM        3          2         1      2.000000

WINNER

🏆 Best Model: XGBoost

✓ Saved: model_comparison.png
✓ Saved: model_comparison_summary.csv

✓ COMPARISON COMPLETE!
